In [18]:
import pandas as pd
from google.cloud import bigquery
import json
import datetime
from queries import *


class APIClient():
    _client = None
    project_name = None

    DAYS_IN_MONTH = {
    1: 31,   # Январь
    2: 28,   # Февраль (оставляем 28 дней)
    3: 31,   # Март
    4: 30,   # Апрель
    5: 31,   # Май
    6: 30,   # Июнь
    7: 31,   # Июль
    8: 31,   # Август
    9: 30,   # Сентябрь
    10: 31,  # Октябрь
    11: 30,  # Ноябрь
    12: 31   # Декабрь
}

    def __init__(self, path):
        self.path_to_service_account = path

    @property
    def path_to_service_account(self):
        return self.__path_to_service_account

    @path_to_service_account.setter
    def path_to_service_account(self, value):
        if not isinstance(value, str):
            raise TypeError("")
        try:
            self._client = bigquery.Client.from_service_account_json(value)
            with open(value, 'r') as file:
                json_data = file.read()
            data = json.loads(json_data)
            self.project_name = data["project_id"]
            self.__path_to_service_account = value
        except:
            print("Некорректный путь")

    def get_event_by_year(self, event_type, year):
        if not self._client:
            return
        query = GET_EVENT_BY_DAY_QUERY

        job_config = bigquery.QueryJobConfig(
            query_parameters=[
                bigquery.ScalarQueryParameter("event_type", "STRING", event_type)
            ]
        )   
        data = self._client.query(
            query % year, 
            job_config=job_config, 
            project=self.project_name)
        
        return data.to_dataframe()
    
    # 20160201
    def get_all_data(self, data_period):
        if isinstance(data_period, int):
            data_period = str(data_period)
        data_period = data_period.strip()
        # year data
        if len(data_period == 4) and self._validate_year_date:
            pass
        elif len(data_period) == 6 and self._validate_month_date:
            pass
        elif len(data_period) == 8 and self._validate_day_date:
            pass
        else:
            raise Exception("Incorrect date format")


    def _validate_year_date(self, date: str) -> bool:
        year = date[0:4]
        result = 2015 <= int(year) <= datetime.datetime.now().year
        return result
    
    
    def _validate_month_date(self, date: str) -> bool:
        if not self._validate_year_date(date):
            return False
        case1 = int(date[4]) == 0 and 1 <= int(date[5]) <= 9
        case2 = int(date[4]) == 1 and 0 <= int(date[5]) <= 2
        return case1 or case2
    

    def _validate_day_date(self, date: str) -> bool:
        if not (self._validate_year_date(date) and self._validate_month_date(date)):
            return False
        
        year = date[0:4]
        month = date[4:6]
        day = date[6:8]
        days_in_month = self.DAYS_IN_MONTH[int(month)]

        if int(month) == 2 and int(year) % 4 == 0:
            days_in_month += 1

        return 0 <= int(day) <= days_in_month
    

In [38]:
path = "auth/bionic-genre-407516-2d1cd98c3fe3.json"

api = APIClient(path)
# api.get_event_by_year("PushEvent", 20190101)


True

In [1]:
import requests
import os

def save_https_request(url):
    try:
        # Выполняем HTTPS запрос
        response = requests.get(url)
        
        if response.status_code == 200:
            # Получаем имя файла из URL
            file_name = url.split('/')[-1]
            
            # Определяем путь к текущей директории
            current_directory = os.path.dirname(os.path.abspath(__file__))
            
            # Сохраняем ответ в файл в текущей директории
            with open(os.path.join(current_directory, file_name), 'wb') as file:
                file.write(response.content)
                print(f"Файл сохранен по пути: {os.path.join(current_directory, file_name)}")
        else:
            print("Ошибка при получении данных:", response.status_code)
    
    except requests.RequestException as e:
        print("Ошибка при выполнении запроса:", e)

# Пример использования функции
url_to_request = 'https://data.gharchive.org/2015-01-01-{0..23}.json.gz'  # замените на нужный URL
save_https_request(url_to_request)

Ошибка при получении данных: 404
